# Create Feature01.csv

產生是否為玉山帳戶、是否為alert 帳戶的特徵 
 
Output: `feature01.csv`  

Form:
| acct | is_ESUN | label |
| --- | --- | --- |

Descriptions:
| Column | Description | Range |
| --- | --- | --- |
| is_ESUN | 是否為玉山帳戶 | {0, 1} |
| label | 是否為 alert 帳戶 | {0, 1} |


In [ ]:
import pandas as pd

### Read files

In [ ]:
column_types = {
    "from_acct": "category",
    "from_acct_type": "category",
    "to_acct": "category",
    "to_acct_type": "category",
    "is_self_txn": "category",
    "txn_amt": "float",
    "txn_date": "category",
    "txn_time": "category",
    "currency_type": "category",
    "channel_type": "category"
}
alert_types = {
    "acct": "category",
    "event_date": "category"
}

df_txn = pd.read_csv("src/acct_transaction.csv", dtype=column_types)
df_alert = pd.read_csv("src/acct_alert.csv", dtype=alert_types)

print("CSV files read complete.")

### 處理資料

In [ ]:
# 合併"from" 和 "to"
from_df = df_txn[["from_acct", "from_acct_type"]].rename(
    columns={"from_acct": "acct", "from_acct_type": "acct_type"}
)
to_df = df_txn[["to_acct", "to_acct_type"]].rename(
    columns={"to_acct": "acct", "to_acct_type": "acct_type"}
)
acct_df = pd.concat([from_df, to_df], ignore_index=True).drop_duplicates("acct")

# 新增 is_ESUN 欄位
acct_df["is_ESUN"] = acct_df["acct_type"].map({"01": 1, "02": 0}).fillna(0).astype(int)

# 加入 label
acct_df["label"] = acct_df["acct"].isin(df_alert["acct"]).astype(int)

# 保留欄位
result = acct_df[["acct", "is_ESUN", "label"]]

# 排序
result = result.sort_values("acct").reset_index(drop=True)

### 輸出

In [ ]:
result.to_csv("feature01.csv", index=False)